In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
images_path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
print("Images downloaded to:", images_path)

Using Colab cache for faster access to the 'nih-chest-x-ray-14-224x224-resized' dataset.
Images downloaded to: /kaggle/input/nih-chest-x-ray-14-224x224-resized


In [7]:
import os

def looks_like_full_label_file(csv_path, min_rows=50000):

    try:
        df = pd.read_csv(csv_path, nrows=5)
    except Exception:
        return False
    if "Finding Labels" not in df.columns:
        return False
    # Cheap row count without loading the whole file
    with open(csv_path) as f:
        row_count = sum(1 for _ in f) - 1  # minus header
    return row_count >= min_rows

import pandas as pd

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(images_path) for f in files if f.endswith(".csv")]
print("CSV files found bundled with images:", csv_candidates)

labels_csv_path = None
for candidate in csv_candidates:
    if looks_like_full_label_file(candidate):
        labels_csv_path = candidate
        print(f"Using bundled labels file: {candidate}")
        break
    else:
        print(f"Rejected {candidate} — doesn't look like the full label set (wrong columns or too few rows)")

if labels_csv_path is None:
    print("No valid labels file bundled with the resized images — fetching Data_Entry_2017.csv "
          "from the original dataset instead (not the full 42GB of images, just this one file).")
    labels_dir = kagglehub.dataset_download("nih-chest-xrays/data", path="Data_Entry_2017.csv")
    labels_csv_path = labels_dir if labels_dir.endswith(".csv") else os.path.join(labels_dir, "Data_Entry_2017.csv")
    print("Labels downloaded to:", labels_csv_path)

img_dir_candidates = [root for root, dirs, files in os.walk(images_path) if any(f.lower().endswith((".png", ".jpg")) for f in files)]
IMAGES_DIR = img_dir_candidates[0]
print("Images folder:", IMAGES_DIR)

CSV files found bundled with images: ['/kaggle/input/nih-chest-x-ray-14-224x224-resized/BBox_List_2017_Official_NIH.csv', '/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv']
Rejected /kaggle/input/nih-chest-x-ray-14-224x224-resized/BBox_List_2017_Official_NIH.csv — doesn't look like the full label set (wrong columns or too few rows)
Using bundled labels file: /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
Images folder: /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224


In [18]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

base_model = DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CONDITIONS), activation="sigmoid")(x)
model = models.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │        14,350 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,051,854 (26.90 MB)

 Trainable params: 14,350 (56.05 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [19]:
def compute_class_weights(labels):

    n = labels.shape[0]
    pos_counts = labels.sum(axis=0)
    neg_counts = n - pos_counts
    pos_weights = (neg_counts / n).astype("float32")
    neg_weights = (pos_counts / n).astype("float32")
    return pos_weights, neg_weights

pos_weights, neg_weights = compute_class_weights(train_labels)

print(f"{'Condition':<20} {'pos_weight':>10} {'neg_weight':>10}")
for i, c in enumerate(CONDITIONS):
    print(f"{c:<20} {pos_weights[i]:>10.4f} {neg_weights[i]:>10.4f}")

Condition            pos_weight neg_weight
Atelectasis              0.9004     0.0996
Cardiomegaly             0.9755     0.0245
Effusion                 0.8838     0.1162
Infiltration             0.8249     0.1752
Mass                     0.9485     0.0514
Nodule                   0.9423     0.0577
Pneumonia                0.9872     0.0127
Pneumothorax             0.9517     0.0483
Consolidation            0.9588     0.0412
Edema                    0.9803     0.0198
Emphysema                0.9778     0.0222
Fibrosis                 0.9855     0.0145
Pleural_Thickening       0.9704     0.0295
Hernia                   0.9980     0.0021


In [20]:
import tensorflow as tf

def get_weighted_loss(pos_weights, neg_weights, epsilon=1e-7):
    pos_weights_t = tf.constant(pos_weights)
    neg_weights_t = tf.constant(neg_weights)

    def weighted_loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
        loss = -(
            pos_weights_t * y_true * tf.math.log(y_pred)
            + neg_weights_t * (1 - y_true) * tf.math.log(1 - y_pred)
        )
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

    return weighted_loss

weighted_loss_fn = get_weighted_loss(pos_weights, neg_weights)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model.compile(
    optimizer="adam",
    loss=weighted_loss_fn,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase1.keras", save_best_only=True, monitor="val_loss"),
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 101s 125ms/step - auc: 0.6021 - loss: 0.9089 - val_auc: 0.6994 - val_loss: 0.8139
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - auc: 0.6776 - loss: 0.8416 - val_auc: 0.7157 - val_loss: 0.8069
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 95ms/step - auc: 0.6982 - loss: 0.8279 - val_auc: 0.7225 - val_loss: 0.7931
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 92ms/step - auc: 0.7085 - loss: 0.8203 - val_auc: 0.7253 - val_loss: 0.8020
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 92ms/step - auc: 0.7196 - loss: 0.8123 - val_auc: 0.7278 - val_loss: 0.7938
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 92ms/step - auc: 0.7202 - loss: 0.8133 - val_auc: 0.7257 - val_loss: 0.8090
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - auc: 0.7272 - loss: 0.8113 - val_auc: 0.7284 - val_loss: 0.7849
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 66s 106ms/step - auc: 0.7284 - loss: 0.8074 - val_auc: 0.7273 - val_loss: 0.7986
Epoch 9/15
625/625 ━━━━━━━━━━━━━━━━━━

In [22]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=weighted_loss_fn,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase2.keras", save_best_only=True, monitor="val_loss"),
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 112s 133ms/step - auc: 0.7261 - loss: 0.9365 - val_auc: 0.7159 - val_loss: 0.8305
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 100ms/step - auc: 0.7364 - loss: 0.8591 - val_auc: 0.7187 - val_loss: 0.8115
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - auc: 0.7363 - loss: 0.8478 - val_auc: 0.7224 - val_loss: 0.8048
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 98ms/step - auc: 0.7434 - loss: 0.8377 - val_auc: 0.7238 - val_loss: 0.8020
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - auc: 0.7453 - loss: 0.8356 - val_auc: 0.7256 - val_loss: 0.7991
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - auc: 0.7488 - loss: 0.8259 - val_auc: 0.7267 - val_loss: 0.7961
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - auc: 0.7527 - loss: 0.8215 - val_auc: 0.7281 - val_loss: 0.7947
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 98ms/step - auc: 0.7548 - loss: 0.8171 - val_auc: 0.7291 - val_loss: 0.7927
Epoch 9/15
625/625 ━━━━━━━━━━━━━━━━━━

In [23]:
from sklearn.metrics import roc_auc_score

# Fill in your actual v2 numbers here for direct comparison
v2_aucs = {
    "Atelectasis": 0.738, "Cardiomegaly": 0.761, "Effusion": 0.814, "Infiltration": 0.651,
    "Mass": 0.687, "Nodule": 0.639, "Pneumonia": 0.650, "Pneumothorax": 0.794,
    "Consolidation": 0.720, "Edema": 0.819, "Emphysema": 0.803, "Fibrosis": 0.721,
    "Pleural_Thickening": 0.677, "Hernia": 0.796,
}

y_true = val_labels
y_pred = model.predict(val_ds, verbose=0)

print(f"{'Condition':<20} {'v3 AUC':>7} {'v2 AUC':>7} {'Change':>8} {'Positives':>10}")
aucs = []
for i, c in enumerate(CONDITIONS):
    n_pos = int(y_true[:, i].sum())
    if n_pos == 0 or n_pos == len(y_true):
        print(f"{c:<20} {'N/A':>7} {v2_aucs.get(c, 0):>7.3f} {'':>8} {n_pos:>10}")
        continue
    auc = roc_auc_score(y_true[:, i], y_pred[:, i])
    aucs.append(auc)
    change = auc - v2_aucs.get(c, 0)
    print(f"{c:<20} {auc:>7.3f} {v2_aucs.get(c, 0):>7.3f} {change:>+8.3f} {n_pos:>10}")

print(f"\nMean AUC (v3): {np.mean(aucs):.3f}   Mean AUC (v2): {np.mean(list(v2_aucs.values())):.3f}")

Condition             v3 AUC  v2 AUC   Change  Positives
Atelectasis            0.738   0.738   +0.000        512
Cardiomegaly           0.758   0.761   -0.003        131
Effusion               0.816   0.814   +0.002        606
Infiltration           0.659   0.651   +0.008        849
Mass                   0.685   0.687   -0.002        230
Nodule                 0.642   0.639   +0.003        277
Pneumonia              0.624   0.650   -0.026         58
Pneumothorax           0.807   0.794   +0.013        212
Consolidation          0.723   0.720   +0.003        173
Edema                  0.827   0.819   +0.008        100
Emphysema              0.805   0.803   +0.002         93
Fibrosis               0.734   0.721   +0.013         83
Pleural_Thickening     0.677   0.677   +0.000        144
Hernia                 0.786   0.796   -0.010         10

Mean AUC (v3): 0.734   Mean AUC (v2): 0.734


In [24]:
import json, os

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v3_weighted_densenet.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

Artifacts written:
total 68356
drwxr-xr-x 2 root root     4096 Sep 23 20:44 .
drwxr-xr-x 1 root root     4096 Sep 23 19:54 ..
-rw-r--r-- 1 root root      219 Sep 23 20:44 condition_names.json
-rw-r--r-- 1 root root 34989400 Sep 23 19:54 v2_densenet_transfer.keras
-rw-r--r-- 1 root root 34989519 Sep 23 20:44 v3_weighted_densenet.keras


In [25]:
import shutil

shutil.make_archive("v3_imaging_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v3_imaging_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>